# Table 4 generator: all-sample failure case analysis

This notebook merges per-sample results from:
- `baseline_planar_results_locality.csv`
- `baseline_planar_largest_hole_results.csv`

and generates:
- full all-sample failure-case table
- top-10 worst failure cases
- markdown table text for writing


In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# =========================
# CONFIG
# =========================
PROJECT_ROOT = Path.cwd().parent
CSV_DIR = PROJECT_ROOT / 'results' / 'csv'
OUT_DIR = CSV_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

RP_AWARE_CSV = CSV_DIR / 'baseline_planar_results_locality.csv'
LARGEST_HOLE_CSV = CSV_DIR / 'baseline_planar_largest_hole_results.csv'

OUT_FULL_CSV = OUT_DIR / 'table4_failure_cases_all_samples.csv'
OUT_FULL_MD = OUT_DIR / 'table4_failure_cases_all_samples.md'
OUT_TOP10_CSV = OUT_DIR / 'table4_failure_cases_top10.csv'

RESIDUAL_THRESHOLD = 1e-6

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CSV_DIR =', CSV_DIR)
print('RP_AWARE_CSV =', RP_AWARE_CSV)
print('LARGEST_HOLE_CSV =', LARGEST_HOLE_CSV)

PROJECT_ROOT = D:\MyJupyter\Works\3Dsegment
CSV_DIR = D:\MyJupyter\Works\3Dsegment\results\csv
RP_AWARE_CSV = D:\MyJupyter\Works\3Dsegment\results\csv\baseline_planar_results_locality.csv
LARGEST_HOLE_CSV = D:\MyJupyter\Works\3Dsegment\results\csv\baseline_planar_largest_hole_results.csv


In [3]:
rp_df = pd.read_csv(RP_AWARE_CSV)
lh_df = pd.read_csv(LARGEST_HOLE_CSV)

print('Removed-part-aware columns:')
print(rp_df.columns.tolist())
print('\nLargest-hole-only columns:')
print(lh_df.columns.tolist())
print('\nRemoved-part-aware shape:', rp_df.shape)
print('Largest-hole-only shape:', lh_df.shape)

Removed-part-aware columns:
['sample_id', 'success', 'reason', 'num_loops_repaired', 'total_loop_len_before', 'nearest_loop_len_after', 'improvement', 'num_new_vertices', 'num_added_faces', 'mean_added_face_quality', 'min_added_face_quality', 'num_added_faces_inside_zone', 'num_added_faces_outside_zone', 'face_locality_ratio']

Largest-hole-only columns:
['sample_id', 'success', 'reason', 'selection_mode', 'num_loops_repaired', 'total_loop_len_before', 'nearest_loop_len_after', 'improvement', 'num_new_vertices', 'num_added_faces', 'mean_added_face_quality', 'min_added_face_quality', 'num_added_faces_inside_zone', 'num_added_faces_outside_zone', 'face_locality_ratio']

Removed-part-aware shape: (10, 14)
Largest-hole-only shape: (10, 15)


In [4]:
def find_first_existing(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f'None of the candidate columns exist: {candidates}')
    return None

sample_col_rp = find_first_existing(rp_df, ['sample_id', 'sample', 'id'])
sample_col_lh = find_first_existing(lh_df, ['sample_id', 'sample', 'id'])

rp_after_col = find_first_existing(rp_df, ['nearest_loop_len_after', 'target_loop_len_after', 'total_loop_len_after'])
lh_after_col = find_first_existing(lh_df, ['nearest_loop_len_after', 'target_loop_len_after', 'total_loop_len_after'])

lh_outside_col = find_first_existing(lh_df, ['num_added_faces_outside_zone', 'added_faces_outside_zone'], required=False)
rp_improve_col = find_first_existing(rp_df, ['improvement'], required=False)
lh_improve_col = find_first_existing(lh_df, ['improvement'], required=False)

print('sample_col_rp  =', sample_col_rp)
print('sample_col_lh  =', sample_col_lh)
print('rp_after_col   =', rp_after_col)
print('lh_after_col   =', lh_after_col)
print('lh_outside_col =', lh_outside_col)

sample_col_rp  = sample_id
sample_col_lh  = sample_id
rp_after_col   = nearest_loop_len_after
lh_after_col   = nearest_loop_len_after
lh_outside_col = num_added_faces_outside_zone


In [5]:
rp_keep = [sample_col_rp, rp_after_col] + ([rp_improve_col] if rp_improve_col else [])
lh_keep = [sample_col_lh, lh_after_col] + ([lh_improve_col] if lh_improve_col else []) + ([lh_outside_col] if lh_outside_col else [])

rp_use = rp_df[rp_keep].copy().rename(columns={
    sample_col_rp: 'sample_id',
    rp_after_col: 'rp_target_loop_len_after',
    **({rp_improve_col: 'rp_improvement'} if rp_improve_col else {})
})

rename_map_lh = {
    sample_col_lh: 'sample_id',
    lh_after_col: 'lh_target_loop_len_after',
}
if lh_improve_col:
    rename_map_lh[lh_improve_col] = 'lh_improvement'
if lh_outside_col:
    rename_map_lh[lh_outside_col] = 'lh_added_faces_outside_zone'

lh_use = lh_df[lh_keep].copy().rename(columns=rename_map_lh)

rp_use['sample_id'] = rp_use['sample_id'].astype(str)
lh_use['sample_id'] = lh_use['sample_id'].astype(str)

merged = pd.merge(rp_use, lh_use, on='sample_id', how='outer')

for c in ['rp_target_loop_len_after', 'lh_target_loop_len_after', 'rp_improvement', 'lh_improvement', 'lh_added_faces_outside_zone']:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors='coerce')

merged.head()

,sample_id,rp_target_loop_len_after,rp_improvement,lh_target_loop_len_after,lh_improvement,lh_added_faces_outside_zone
0,2594,0.0,0.383482,0.383482,1.401152,117
1,35580,0.0,1.172208,0.781440,-0.390672,0
2,35871,0.0,1.461360,1.461360,3.364885,78
3,36074,0.0,0.273811,0.000000,0.273811,31
4,36192,0.0,0.000000,0.000000,0.000000,0


In [6]:
def make_observation(row):
    rp_after = row.get('rp_target_loop_len_after', np.nan)
    lh_after = row.get('lh_target_loop_len_after', np.nan)
    outside = row.get('lh_added_faces_outside_zone', np.nan)

    if pd.isna(lh_after):
        return 'Largest-hole-only result missing.'
    if lh_after <= RESIDUAL_THRESHOLD:
        return 'Largest-hole-only also closes the target opening.'
    if not pd.isna(rp_after) and rp_after <= RESIDUAL_THRESHOLD:
        if not pd.isna(outside) and outside >= 20:
            return 'Largest-hole-only leaves the target opening unresolved and adds many faces outside the repair zone.'
        if lh_after >= 1.0:
            return 'Largest-hole-only fails badly and leaves a very large residual target opening.'
        if lh_after >= 0.3:
            return 'Largest-hole-only repairs the wrong region and leaves a clear residual target opening.'
        return 'Largest-hole-only underperforms on the target opening.'
    return 'Check this sample manually.'

merged['observation'] = merged.apply(make_observation, axis=1)
merged['failure_gap'] = merged['lh_target_loop_len_after'] - merged['rp_target_loop_len_after']
merged = merged.sort_values(['lh_target_loop_len_after', 'failure_gap'], ascending=[False, False], na_position='last').reset_index(drop=True)

merged.head(10)

,sample_id,rp_target_loop_len_after,rp_improvement,lh_target_loop_len_after,lh_improvement,lh_added_faces_outside_zone,observation,failure_gap
0,42103,0.0,1.476778,1.476778,-0.641232,26,Largest-hole-only leaves the target opening un...,1.476778
1,35871,0.0,1.461360,1.461360,3.364885,78,Largest-hole-only leaves the target opening un...,1.461360
2,35580,0.0,1.172208,0.781440,-0.390672,0,Largest-hole-only repairs the wrong region and...,0.781440
3,43618,0.0,0.661716,0.661716,1.418479,58,Largest-hole-only leaves the target opening un...,0.661716
4,37675,0.0,0.571676,0.571676,1.697881,70,Largest-hole-only leaves the target opening un...,0.571676
5,37784,0.0,0.392229,0.392229,-0.243952,32,Largest-hole-only leaves the target opening un...,0.392229
6,2594,0.0,0.383482,0.383482,1.401152,117,Largest-hole-only leaves the target opening un...,0.383482
7,36074,0.0,0.273811,0.000000,0.273811,31,Largest-hole-only also closes the target opening.,0.000000
8,36192,0.0,0.000000,0.000000,0.000000,0,Largest-hole-only also closes the target opening.,0.000000
9,38335,0.0,0.287380,0.000000,0.287380,31,Largest-hole-only also closes the target opening.,0.000000


In [7]:
final_cols = ['sample_id', 'rp_target_loop_len_after', 'lh_target_loop_len_after']
if 'lh_added_faces_outside_zone' in merged.columns:
    final_cols.append('lh_added_faces_outside_zone')
if 'rp_improvement' in merged.columns:
    final_cols.append('rp_improvement')
if 'lh_improvement' in merged.columns:
    final_cols.append('lh_improvement')
final_cols += ['failure_gap', 'observation']

table_all = merged[final_cols].copy()
table_all.head()

,sample_id,rp_target_loop_len_after,lh_target_loop_len_after,lh_added_faces_outside_zone,rp_improvement,lh_improvement,failure_gap,observation
0,42103,0.0,1.476778,26,1.476778,-0.641232,1.476778,Largest-hole-only leaves the target opening un...
1,35871,0.0,1.461360,78,1.461360,3.364885,1.461360,Largest-hole-only leaves the target opening un...
2,35580,0.0,0.781440,0,1.172208,-0.390672,0.781440,Largest-hole-only repairs the wrong region and...
3,43618,0.0,0.661716,58,0.661716,1.418479,0.661716,Largest-hole-only leaves the target opening un...
4,37675,0.0,0.571676,70,0.571676,1.697881,0.571676,Largest-hole-only leaves the target opening un...


In [8]:
table_all.to_csv(OUT_FULL_CSV, index=False)
table_top10 = table_all.head(10).copy()
table_top10.to_csv(OUT_TOP10_CSV, index=False)

print('Saved:', OUT_FULL_CSV)
print('Saved:', OUT_TOP10_CSV)
print('#rows =', len(table_all))

Saved: D:\MyJupyter\Works\3Dsegment\results\csv\table4_failure_cases_all_samples.csv
Saved: D:\MyJupyter\Works\3Dsegment\results\csv\table4_failure_cases_top10.csv
#rows = 10


In [10]:
display_df = table_all.copy()
for c in display_df.columns:
    if c != 'sample_id' and pd.api.types.is_numeric_dtype(display_df[c]):
        display_df[c] = display_df[c].map(lambda x: '' if pd.isna(x) else (f'{x:.6f}' if abs(x) < 10 else f'{x:.3f}'))

md_text = display_df.to_markdown(index=False)
OUT_FULL_MD.write_text(md_text, encoding='utf-8')

print('Saved:', OUT_FULL_MD)
print('\nPreview:')
print(md_text[:2000])

Saved: D:\MyJupyter\Works\3Dsegment\results\csv\table4_failure_cases_all_samples.md

Preview:
|   sample_id |   rp_target_loop_len_after |   lh_target_loop_len_after |   lh_added_faces_outside_zone |   rp_improvement |   lh_improvement |   failure_gap | observation                                                                                         |
|------------:|---------------------------:|---------------------------:|------------------------------:|-----------------:|-----------------:|--------------:|:----------------------------------------------------------------------------------------------------|
|       42103 |                          0 |                   1.47678  |                            26 |         1.47678  |        -0.641232 |      1.47678  | Largest-hole-only leaves the target opening unresolved and adds many faces outside the repair zone. |
|       35871 |                          0 |                   1.46136  |                            78 |         1.4613

In [11]:
table_all.head(20)

,sample_id,rp_target_loop_len_after,lh_target_loop_len_after,lh_added_faces_outside_zone,rp_improvement,lh_improvement,failure_gap,observation
0,42103,0.0,1.476778,26,1.476778,-0.641232,1.476778,Largest-hole-only leaves the target opening un...
1,35871,0.0,1.461360,78,1.461360,3.364885,1.461360,Largest-hole-only leaves the target opening un...
2,35580,0.0,0.781440,0,1.172208,-0.390672,0.781440,Largest-hole-only repairs the wrong region and...
3,43618,0.0,0.661716,58,0.661716,1.418479,0.661716,Largest-hole-only leaves the target opening un...
4,37675,0.0,0.571676,70,0.571676,1.697881,0.571676,Largest-hole-only leaves the target opening un...
5,37784,0.0,0.392229,32,0.392229,-0.243952,0.392229,Largest-hole-only leaves the target opening un...
6,2594,0.0,0.383482,117,0.383482,1.401152,0.383482,Largest-hole-only leaves the target opening un...
7,36074,0.0,0.000000,31,0.273811,0.273811,0.000000,Largest-hole-only also closes the target opening.
8,36192,0.0,0.000000,0,0.000000,0.000000,0.000000,Largest-hole-only also closes the target opening.
9,38335,0.0,0.000000,31,0.287380,0.287380,0.000000,Largest-hole-only also closes the target opening.


Suggested usage:

- `table4_failure_cases_all_samples.csv`: full all-sample table
- `table4_failure_cases_top10.csv`: top-10 worst failure cases
- `table4_failure_cases_all_samples.md`: markdown table text for writing
